# AI-Orchestrator — Treino LoRA Qwen3.5-9B (Fases 2-3)

LoRA **bf16** (QLoRA 4-bit contraindicado pela Unsloth em Qwen3.5) sobre `unsloth/Qwen3.5-9B`,
dataset SFT de tool-calling/routing gerado na Fase 1, export GGUF Q4_K_M para Ollama local.

Runtime: **A100 40GB**. Rode as células em ordem. Pré-requisito no Drive:
`MyDrive/ai-orchestrator-dataset/orch_sft_train.jsonl` e `orch_sft_val.jsonl`.

**IMPORTANTE:** Checkpoints e outputs vão direto pro Drive. Se a sessão cair, o progresso persiste.

In [ ]:
# 1) Instalação pinada — Qwen3.5 exige transformers v5 (doc oficial Unsloth)
# IMPORTANTE: após rodar esta célula, faça Ambiente de execução → Reiniciar sessão
# (NÃO "Desconectar e excluir"), depois rode a célula 1b.
!pip uninstall -y -q torchaudio 2>/dev/null
!pip install -q --no-cache-dir "unsloth" "unsloth_zoo"
!pip install -q "transformers>=5.2.0,<=5.5.0" "trl>=0.18.2,<=0.24.0" "datasets>=3.4.1,<4.4.0" "accelerate>=1.2" "peft>=0.16" sentencepiece "protobuf<7"
print("INSTALADO — agora: Ambiente de execução → Reiniciar sessão, depois rode célula 1b")

In [ ]:
# 1b) Verificação pós-restart — rode APÓS reiniciar a sessão
import unsloth  # primeiro, antes de transformers
import transformers, trl, datasets as ds_lib
print('transformers:', transformers.__version__)
print('trl:', trl.__version__)
print('datasets:', ds_lib.__version__)
assert transformers.__version__.split('.')[0] == '5', 'transformers errado'
print('OK — ambiente pronto')

In [ ]:
# 2) Drive + dataset — leitura manual (Arrow falha ao inferir schema dos tool_calls heterogêneos)
from google.colab import drive
drive.mount('/content/drive')

import json

DATA_DIR = '/content/drive/MyDrive/ai-orchestrator-dataset'

def load_rows(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

raw = {
    'train': load_rows(f'{DATA_DIR}/orch_sft_train.jsonl'),
    'val':   load_rows(f'{DATA_DIR}/orch_sft_val.jsonl'),
}
print({k: len(v) for k, v in raw.items()})
print('exemplo (roles):', [m['role'] for m in raw['train'][0]['messages']])

In [ ]:
# 3) Modelo base + LoRA bf16
# load_in_4bit=False: Unsloth contraindica QLoRA 4-bit em Qwen3.5 (LoRA 16-bit é o recomendado).
# target_modules: conjunto da doc oficial Unsloth p/ Qwen3.5 (atenção + MLP).
# As camadas lineares DeltaNet (híbridas) ficam FORA dos adapters — mesma escolha da doc/plano.
import torch
from unsloth import FastLanguageModel

MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen3.5-9B",
    max_seq_length  = MAX_SEQ,
    dtype           = torch.bfloat16,
    load_in_4bit    = False,
    load_in_16bit   = True,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
    max_seq_length = MAX_SEQ,
)
model.print_trainable_parameters()

In [ ]:
# 4) Formatação do dataset — aplica chat template e converte pra Dataset HF
from datasets import Dataset

def format_row(row):
    msgs = []
    for m in row['messages']:
        mc = dict(m)
        if not isinstance(mc.get('content'), str):
            mc['content'] = json.dumps(mc.get('content'), ensure_ascii=False)
        msgs.append(mc)
    return tokenizer.apply_chat_template(msgs, tokenize=False)

dataset = {
    'train': Dataset.from_dict({'text': [format_row(r) for r in raw['train']]}),
    'val':   Dataset.from_dict({'text': [format_row(r) for r in raw['val']]}),
}
from datasets import DatasetDict
dataset = DatasetDict(dataset)
print(dataset)

In [ ]:
# 5) SFTTrainer — 2 epochs, lr 2e-4 cosine, batch efetivo 16, eval por epoch
# IMPORTANTE: output_dir no Drive — checkpoints sobrevivem reset de sessão.
from trl import SFTTrainer, SFTConfig
import os

DRIVE_OUT = '/content/drive/MyDrive/ai-orchestrator-lora/training'
os.makedirs(DRIVE_OUT, exist_ok=True)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset  = dataset["val"],
    args = SFTConfig(
        dataset_text_field          = "text",
        max_length                  = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,   # batch efetivo 16
        num_train_epochs            = 2,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.03,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        bf16                        = True,
        logging_steps               = 10,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        output_dir                  = DRIVE_OUT,  # Drive, não disco efêmero
        seed                        = 42,
        report_to                   = "none",
    ),
)

# Mascara loss fora dos turnos do assistant (tool results viram turno "user"
# no template ChatML do Qwen, logo também ficam mascarados — comportamento desejado).
try:
    from unsloth.chat_templates import train_on_responses_only
    if "<|im_start|>" in (tokenizer.chat_template or ""):
        trainer = train_on_responses_only(
            trainer,
            instruction_part = "<|im_start|>user\n",
            response_part    = "<|im_start|>assistant\n",
        )
        print("train_on_responses_only: ATIVO")
    else:
        print("AVISO: template sem marcadores ChatML — treinando sequência completa")
except Exception as exc:
    print(f"AVISO: train_on_responses_only indisponível ({exc}) — treinando sequência completa")

print(f"Checkpoints serão salvos em: {DRIVE_OUT}")

In [ ]:
# 6) Treino + métricas + VRAM
# NOTA: trainer.evaluate() pós-treino causa CUDA IllegalMemoryAccess nas camadas
# DeltaNet do Qwen3.5 (bug Unsloth em inferência). Val loss confiável já é computado
# durante o treino (eval_strategy="epoch"). Não chamar evaluate() separadamente.
import torch

torch.cuda.reset_peak_memory_stats()
stats = trainer.train()

print(f"train_loss final: {stats.metrics.get('train_loss'):.4f}")
print(f"tempo: {stats.metrics.get('train_runtime', 0)/60:.1f} min")

# Histórico train/val loss por step (overfit check: val subindo entre epochs = parar)
for row in trainer.state.log_history:
    if "loss" in row or "eval_loss" in row:
        print(row)

vram = torch.cuda.max_memory_reserved() / 1024**3
print(f"VRAM pico: {vram:.1f} GB / 40 GB (esperado ~22-26 GB p/ 9B bf16 LoRA)")

In [ ]:
# 7a) Limpeza de disco — libera espaço para merge + GGUF (~25 GB necessários)
# Cache HuggingFace contém modelo base (~19 GB) que já está carregado em memória.
# Checkpoints já estão seguros no Drive.
import shutil, os

# Remove cache HF (modelo base já carregado em VRAM)
hf_cache = '/root/.cache/huggingface/hub'
if os.path.exists(hf_cache):
    sz = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(hf_cache) for f in fn)
    shutil.rmtree(hf_cache, ignore_errors=True)
    print(f'Cache HF removido: {sz/1024**3:.1f} GB liberados')
else:
    print('Cache HF já limpo')

# Mostra espaço livre
os.system('df -h /content | tail -1')

In [ ]:
# 7b) Export: merge 16-bit -> GGUF Q4_K_M -> Drive (+Modelfile Ollama)
import glob, os, shutil

DST = '/content/drive/MyDrive/ai-orchestrator-lora'
os.makedirs(DST, exist_ok=True)

# Merge dos adapters no modelo base em 16-bit
model.save_pretrained_merged('/content/merged', tokenizer, save_method='merged_16bit')

# Conversão GGUF Q4_K_M (Unsloth compila llama.cpp na primeira chamada)
# NOTA: Unsloth gera em /content/gguf_gguf/ (adiciona _gguf ao path informado)
model.save_pretrained_gguf('/content/gguf', tokenizer, quantization_method='q4_k_m')

ggufs = sorted(
    glob.glob('/content/gguf_gguf/**/*.gguf', recursive=True) +
    glob.glob('/content/gguf/**/*.gguf', recursive=True),
    key=os.path.getsize, reverse=True)
assert ggufs, 'nenhum .gguf gerado — cheque /content/gguf e /content/gguf_gguf'
src = [g for g in ggufs if 'Q4_K_M' in g][0]
gguf_name = 'qwen3.5-9b-orch.Q4_K_M.gguf'
shutil.copy2(src, f'{DST}/{gguf_name}')
print(f'GGUF copiado: {DST}/{gguf_name} ({os.path.getsize(src)/1024**3:.1f} GB)')

GO_TEMPLATE = '''{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{ .System }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}
{{- end }}
{{- if .ToolCalls }}<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}'''

TQ = '"' * 3
modelfile = (
    f'FROM ./{gguf_name}\n\n'
    f'TEMPLATE {TQ}{GO_TEMPLATE}{TQ}\n\n'
    'PARAMETER stop "<|im_start|>"\n'
    'PARAMETER stop "<|im_end|>"\n'
    'PARAMETER temperature 0.7\n'
    'PARAMETER top_p 0.8\n'
    'PARAMETER top_k 20\n'
    'PARAMETER repeat_penalty 1.0\n'
    'PARAMETER num_ctx 8192\n'
)
with open(f'{DST}/Modelfile', 'w') as fh:
    fh.write(modelfile)
print(f'Modelfile escrito em {DST}/Modelfile')

## 8) Deploy local (RTX 3060 / Ollama)

1. Baixe do Drive (`MyDrive/ai-orchestrator-lora/`) o `.gguf` (~5.5 GB) e o `Modelfile`
   para o mesmo diretório local.
2. Crie o modelo no Ollama:
   ```bash
   ollama create qwen3.5-9b-orch -f Modelfile
   ```
3. Rode os 3 gates (watchdog ativo) a partir da raiz do projeto:
   ```bash
   MODEL=qwen3.5-9b-orch python evals/eval_routing.py    # gate: >=90%
   MODEL=qwen3.5-9b-orch python evals/eval_injection.py  # gate: 0 leaks
   MODEL=qwen3.5-9b-orch python evals/eval_domains.py    # gate: >=80% por domínio
   ```
4. **Critério de adoção** (Fase 4 do plano): superar 87.5% em domains **sem regressão**
   em routing/injection. Se empatar ou piorar, manter o 9B base e documentar o
   experimento no README. Promoção (Fase 5): `.env MODEL=qwen3.5-9b-orch` + restart.